# 🫀 퀘스트 46 · Q7-Z — **탈감쇠 모형을 직접 검정한다**

| | **MedKOS / `notebooks/quest46_q7z_direct_test.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-X**(λ(Δ) · null · 동등분산) · **Q7-S″**(W1·W5) · **Q7-V**(λ 0.2586) |
| 규약 | **R11 · R16 · R22 · R29 ② · R30 ① · R33 ① · R34 ②③ · R35 ①④⑦ · R36 ①⑤ · R37 · R38** |
| 학습 | **0회** · GPU 불필요 · **새 데이터 0** · 예상 20~35분 |

## 왜 Q7-Y 가 아니라 이 런인가

지금 결론을 떠받치는 사슬은 이렇다.

```
V1  λ 가 0.26 으로 낮다
X1  분자를 적분으로 바꾸면 λ 가 0.57 까지 오른다
      ↓  ← ★ **이 화살표가 검정되지 않았다**
X6  그러면 효과가 두 배가 되고
      ↓
    「필요표본 124 → 31. SVDB 안에서 끝난다」
```

가운데 화살표는 **탈감쇠 모형**(선형성 · 오차 독립 · 이분정규)이다. X0 이 그걸
스스로 인정했다 — 「진짜 전제는 선형성과 오차 독립성인데 그건 안 쟀습니다」.
**Q7-Y 의 설계 전제가 바로 이 모형**이므로, 검정 없이 Q7-Y 를 지으면
안 될 수도 있는 것 위에 반나절을 더 쌓는다.

**직접 검정이 훨씬 싸다.** `p_score` 대신 적분 분자를 넣고 S3 를 다시 재면,
모형이 예측한 만큼 오르는지 **한 번에** 확인된다. 그리고 이 검정은 AUROC↔d
정규성 가정까지 **우회**한다 — 예측값과 실측을 비교하는 것이므로 변환의 타당성이
자동으로 검정된다.

## ★★ 예측을 **비**로 사전등록하는 이유

`p_energy` 는 파형이 필요한데 **비트 절단 배열에는 왼쪽 문맥이 없다**
(Q7-V 1판이 여기서 죽었다 — 자산 대비 corr 0.7122). SVDB 연속 신호를 다시 받는 건
비싸다. 그래서 **두 통계량을 똑같이 비트 배열에서** 계산하고, 예측을 **절대값이 아니라
비**로 건다.

```
사전등록 예측:   초과(w) / 초과(w=0)  ≈  λ_w / λ_0  ≈  0.5694 / 0.2891  ≈ 1.97   (w=40ms)
```

기준선이 자산값(0.5362)에서 움직여도 **대비는 유효**하다. 그리고 `w=0` 이 곧
`p_score` 이므로 **w 축 하나만 바뀐다**(분모·창 동일).

⚠️ **자산 대비 corr ≈ 0.71 을 재현하는지**가 그 차이를 우리가 이해하고 있다는 증명이다
(Z0). 재현 못 하면 다른 것이 또 어긋난 것이므로 **중단**한다.

## 외부 검토가 지적한 나머지 — 같은 런에 묶는다

- **Z2** `null 0.5090` 은 **기전이 아직 규명되지 않았다.** 「MC 잡음」설명은 틀렸다 —
  MC 잡음은 중심을 안 옮기고 CI 를 **넓힌다**. 후보 하나: `matched_auc` 는
  `tot < MIN_PAIR` 에서 nan 을 내는데 치환하면 `tot = Σ n_S(k)·n_N(k)` 가 **바뀌고**,
  `np.nanmean` 이 살아남은 것만 평균하면 **선택 효과**가 생긴다. `reps` 사다리에
  **nan 드롭 수**를 같이 찍어 가른다. ★ 확정 전까지 필요표본은 **213/435 와 124/253
  을 병기**한다
- **Z3** 128Hz 라운딩이 `λ(3ms)` 만이 아니라 **`λ(11ms)` 도 오염**한다 — 128Hz 에서
  Δ=5 와 Δ=11 이 **같은 1샘플(7.8ms)** 이다. 그리고 **S3 를 재는 SVDB 가 128Hz** 이므로
  X6 의 나눗셈 계수는 **128Hz 층의 λ** 여야 한다. Δ 를 **샘플 단위**로 다시 잡고
  표본율로 층화한다. 덤으로 **실측 오차 분포 하** λ_w 를 재서 λ_energy 의 **짝을
  맞춘다**(X1 의 λ_energy 는 고정 11ms 였다)
- **Z4** X4 의 `ρ(쌍 수)=+0.3360` 에 **라벨셔플 영점**을 붙인다
- **Z5 ★** **결론 검산표** — 판정마다 (a) 근거 숫자 (b) 그 숫자가 의존하는 **미검정
  가정** (c) 그 가정이 틀렸을 때 결론이 어떻게 바뀌는지. 지금까지 대화로 하던 걸
  **코드에 고정**한다(R38 ①)

## ★ 정정 — 「λ 붕괴 = 순수 위치 민감도」는 성립하지 않는다

잡음이 없고 매끄러운 P 를 점으로 샘플링하는 것뿐이라면, 비트 `i` 의 값은
`A_i·g(0)`, 이동 후는 `A_i·g(Δ)` 로 **둘 다 `A_i` 에 비례**하므로 비트 간 상관은
**Δ 와 무관하게 1** 이다.

> **즉 순수한 위치 민감도만으로는 λ 가 떨어질 수 없다.** λ(11ms)=0.2891 은
> **위치에 따라 탈상관되는 비트별 성분 = 잡음**이 있어야만 나온다.

「지터 한계」와 「잡음 지배」는 대립 가설이 아니라 **같은 현상의 두 이름**이다.
그리고 이게 Q7-Y 설계를 가른다 — **잡음 지배면 구간 최댓값은 잡음의 최댓값을
고른다**(양의 편의 · P 부재 28.8% 비트에서 최악). 좋은 일은 **적분**이 하고
max 는 위험만 얹는다. 그래서 Q7-Y 는 **팔 A(최댓값) / 팔 B(고정 위치) 병행**이어야 한다.

## 사전등록 — 관문

| 관문 | 판정 기준 |
|---|---|
| **Z0** | ★ **구성적 항등** — 창이 온전한 비트에서 corr ≥ 0.99 · 전체보다 높음 |
| **Z1 ★★★(주)** | 초과비 `w=40 / w=0` 가 **1.97 ± MDE 환산** 안인가 |
| **Z2 ★★** | `reps` 를 늘리면 null 이 0.5 로 **수렴**하나 · nan 드롭이 설명하나 |
| **Z3 ★★** | 표본율 층별 λ · **샘플 단위** Δ · 실측 오차 하 λ_w |
| **Z4** | `ρ(쌍 수)` 가 **라벨셔플 영점**을 넘나 |
| **Z5** | 결론 검산표 — 미검정 가정을 **빠짐없이** 나열 |

### 판정표 (R29 ②)

- **Z0 실패** → 좌표 이해가 또 틀렸다. **어떤 관문도 읽지 않는다**
- **Z1 비 ≈ 1.97** → ★ **탈감쇠 모형 유효.** Q7-Y 진행(팔 A/B 병행 · w 는 Z1 이 정함)
- **Z1 비 ≈ 1.0** → ★★ **λ 와 효용이 연결되지 않는다.** X6 상한 전부 무효 ·
  「31/63」 폐기 · **「자가 병목」 결론 재검토** · 갈래 종결 검토
- **Z1 중간** → 회복률을 **실측값**으로 X6 재작성
- **Z2 미규명** → 필요표본을 **두 값 병기**로 유지한다

⚠️ **이 런은 SVEB 질문에 답하지 않는다.** 모형을 검정한다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    """**레코드 단위** 부트스트랩(R11)."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, fn, seed, nb=2000, q=2.5):
    """짝을 유지한 레코드 부트스트랩 — 비·차 같은 유도량용."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    ok_ = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok_], b[ok_]
    if len(a) < 5:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    out = []
    for _ in range(nb):
        j = rng.randint(0, len(a), len(a))
        try:
            out.append(float(fn(a[j], b[j])))
        except Exception:
            pass
    if len(out) < 50:
        return float(fn(a, b)), float("nan"), float("nan"), len(a)
    return (float(fn(a, b)), float(np.percentile(out, q)),
            float(np.percentile(out, 100 - q)), len(a))

def spearman(a, b):
    ra = np.asarray(a, float).argsort().argsort().astype(float)
    rb = np.asarray(b, float).argsort().argsort().astype(float)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300

BUT_DIR = "but-pdb/1.0.0"
DELIN, INPUT = "dwt", "raw"
TOL_MS = 50.0
P_LO_MS, P_HI_MS = -278.0, -42.0
SCORE_HALF_MS = 100.0                      # 분모(MAD) 창 — **전 팔 공통**
MIN_P = 5
REC_EXPECT, REC_FLOOR = 48, 40   # ★ Q7-U·V·X 가 쓴 레코드 수 · 이보다 적으면 중단
NET_TRIES, NET_BASE = 5, 2.0        # 재시도 · 지수 백오프(2·4·8·16초)

# ── ★★★ Z1 — 적분 반폭 격자. **w=0 이 `p_score`** (분자가 단일 표본)
W_MS = (0.0, 10.0, 20.0, 30.0, 40.0, 60.0, 80.0, 100.0)
W_KEY = 40.0                               # X1 이 쓴 값 — 예측의 기준점
RATIO_PRED = 1.97                          # λ_energy/λ_score = 0.5694/0.2891

# ── Z3 — Δ 를 **샘플 단위**로. 128Hz 에서 1샘플 = 7.8125ms · 360Hz 에서 2.78ms
D_SAMP = (0, 1, 2, 3, 4, 6, 8, 12)
D_KEY_S = 2                                # 판정 기준 샘플 수

# ── Z2 — null 사다리
REPS = (3, 5, 10, 20, 50)

QUANT_MS = 1000.0 / 128.0
FULL_K = tuple(range(4, 33))
MIN_S, MIN_N, MIN_PAIR = 25, 25, 200

SV5  = os.path.join(MITBIH, "svdb_data5.npz")
PDEL = os.path.join(MITBIH, "svdb_pdelin.npz")

# ── ★ 앞선 공식 실행값
REF = dict(
    # ⚠️ **Q7-V 1판의 0.7122 를 기준으로 쓰지 않는다** — 그 값은 창 중심을 `np.clip`
    #    으로 **옮기던 버그** 코드의 출력이다(그래서 갈아엎었다). Q7-P0 의 실제 규약은
    #    **창을 자르고 중심은 유지**하는 것이고, 그러면 corr 이 훨씬 높게 나온다(실측 0.9312).
    #    → 기준을 「그 숫자」가 아니라 **구성적 항등**으로 바꾼다(아래 Z0).
    asset_corr_v1bug=0.7122,
    s3_asset=0.5362, s3_lo=0.4841, s3_hi=0.5903, s3_n=34, s3_mde=0.0531,
    null_s2=0.5090, null_x3=0.5005,        # Q7-S′ vs Q7-X — **아직 안 갈렸다**
    lam_score=0.2891, lam_energy=0.5694,   # X1 @Δ=11ms
    lam_v=0.2586,                          # V1 실측 오차 분포 하
    w1=dict(mean=0.0048, lo=-0.0030, hi=0.0141, n=56),
    rho_pairs=0.3360,                      # X4
    need=dict(null_old=(213, 435), null_new=(124, 253)))   # ★ 병기한다
FIT_CORR_MIN = 0.99        # ★ 창이 **온전히 들어가는** 비트에서 요구하는 corr
FIT_MED_MAX  = 0.05        # 그 비트들의 |Δ| 중앙 상한 (npz 는 float32 · 자산은 float64)

RULE_CHECK = {
    "R11 환자 단위":      "부트스트랩·판정 전부 **레코드 단위**",
    "R16 fallback 없음":  "자산 없으면 **중단**",
    "R22 누수 없음":      "λ·오차 분포는 BUT PDB 에서, SVDB 라벨은 안 본다",
    "R29 ② 분기 금지":    "Z0 이 깨지면 어떤 관문도 읽지 않는다",
    "R30 ① 필요표본":     "★★ null 이 안 갈렸으므로 **두 값을 병기**한다",
    "R33 ① MDE":          "예측 검정도 MDE 로 환산해 비교",
    "R34 ② 선택 편의":    "w 는 **사전등록 격자** — 성적표에서 고르지 않는다",
    "R34 ③ 대조 보장":    "★ w=0 이 곧 `p_score` — **구성으로** 기준선이다",
    "R35 ④ 항등 대조":    "★ w=0 재계산이 자산과 **corr 0.71 을 재현**해야 한다",
    "R36 ① 상한":         "탈감쇠는 상한 · 점추정은 **CI 와 함께만**",
    "R36 ⑤ 성분 병기":    "비는 **분자·분모 값과 함께만** 인용",
    "R38 ① 검산":         "★★ **결론 검산표**(Z5)를 코드에 고정",
}

CONFIG = dict(
    exp="quest46_q7z_direct_test", quest="ailab-2026-0046", step="deattenuation-test",
    parent_exp=["quest46_q7x_diagnostics", "quest46_q7s3_recompute",
                "quest46_q7v_ruler_audit"],
    purpose=("**탈감쇠 모형을 직접 검정한다.** V1(λ 낮다)+X1(적분으로 λ 두 배)에서 "
             "X6(효과 두 배 · 필요표본 1/4)로 가는 화살표가 미검정이고, **Q7-Y 의 설계 "
             "전제가 바로 그것**이다. `p_score` 대신 적분 분자를 넣고 S3 를 다시 재면 "
             "모형이 예측한 만큼 오르는지 한 번에 확인되며, AUROC↔d 정규성 가정까지 "
             "**우회**한다. ★ 파형이 필요한데 비트 절단엔 왼쪽 문맥이 없으므로(Q7-V 1판 "
             "corr 0.7122), 두 통계량을 **똑같이 비트 배열에서** 계산하고 예측을 "
             "**비**로 건다. 덤으로 외부 검토가 지적한 넷을 같은 런에 묶는다 — null "
             "기전 · 128Hz 라운딩(λ(11ms)도 오염) · λ_energy 의 짝 · 결론 검산표"),
    dataset="BUT PDB 50×2분(전문가 P 주석) + SVDB 78레코드 184,499비트",
    w_ms=list(W_MS), w_key=W_KEY, ratio_pred=RATIO_PRED, d_samp=list(D_SAMP),
    d_key_s=D_KEY_S, reps=list(REPS), ref=REF, fit_corr_min=FIT_CORR_MIN, fit_med_max=FIT_MED_MAX,
    rule_check=RULE_CHECK,
    predictions={
        "Z0": f"★ **구성적 항등 증명**(숫자 재현이 아니다). 창이 비트 배열에 **온전히 "
              f"들어가는** 비트(`HW ≤ p_idx ≤ L−HW−1`)는 연속 신호 창과 **같은 표본**을 "
              f"담으므로 자산과 반드시 일치해야 한다 — corr ≥ {FIT_CORR_MIN} · |Δ| 중앙 "
              f"≤ {FIT_MED_MAX} · 그리고 전체 corr 보다 **높아야** 한다(불일치가 잘린 "
              "비트에만 있다는 뜻). ⚠️ Q7-V 1판의 0.7122 는 창 중심을 옮기던 **버그** "
              "코드의 출력이라 기준으로 쓰지 않는다 — **버그 있는 코드의 출력에 재현 "
              "기준을 앵커하면 안 된다**. 아니면 **중단**",
        "Z1": f"★★★ **(주) 적분 반폭 w 스윕.** w∈{W_MS} ms 로 S3 를 다시 잰다. "
              f"**w=0 이 곧 `p_score`**(분자 단일 표본)이고 분모·창은 전부 동일하다. "
              f"사전등록 예측 — 초과비 `w={W_KEY:.0f} / w=0` ≈ **{RATIO_PRED}** "
              f"(= λ_energy/λ_score = {REF['lam_energy']}/{REF['lam_score']}). "
              "비가 1 근처면 **λ 와 효용이 연결되지 않는다**",
        "Z2": f"**null 기전.** `reps`∈{REPS} 로 나란히 재고 **nan 드롭 수**를 같이 찍는다. "
              "「MC 잡음」설명은 틀렸다(중심을 안 옮기고 CI 를 넓힌다). 후보: "
              "`matched_auc` 가 `tot<MIN_PAIR` 에서 nan 을 내는데 치환하면 `tot` 가 "
              "바뀌고, `nanmean` 이 살아남은 것만 평균하면 **선택 효과**가 생긴다",
        "Z3": f"**λ 를 표본율로 층화 + Δ 를 샘플 단위**({D_SAMP}). 128Hz 에서 Δ=5ms 와 "
              "11ms 가 **같은 1샘플**이라 λ(11ms) 가 위로 편향됐다. **SVDB 가 128Hz** 이므로 "
              "X6 계수는 128Hz 층 값이어야 한다. 덤으로 **실측 오차 분포 하** λ_w 를 재서 "
              "X1 의 고정-Δ λ_energy 에 **짝을 맞춘다**",
        "Z4": f"X4 의 `ρ(쌍 수)={REF['rho_pairs']}` 에 **라벨셔플 영점**을 붙인다 — "
              "쌍 수와 AUROC 의 연관이 인공물인지 가른다",
        "Z5": "★ **결론 검산표.** 판정마다 (a) 근거 숫자 (b) 그 숫자가 의존하는 **미검정 "
              "가정** (c) 그 가정이 틀렸을 때 결론이 어떻게 바뀌는지를 **코드가** 찍는다"},
    caveat=("★ **새 데이터 0** · 학습 0회 · SVEB 질문에 답하지 않는다 — **모형을 검정한다**. "
            "★ **필요표본은 두 값 병기**(213/435 · 124/253) — null 기전이 안 갈렸다. "
            "★ `p_score_beat` 는 창이 **잘리는** 비트에서 자산과 다르다(연속 신호에 있던 왼쪽 문맥이 비트 배열엔 없다). 그래서 Z0 은 **창이 온전한 비트**에서 항등을 증명하고, "
            "그래서 **절대값이 아니라 비**를 사전등록했고, w=0 을 **같은 계산**의 기준선으로 "
            "쓴다. 자산값(0.5362)과 직접 비교하지 않는다. "
            "★ **W1 은 5차원 중 1축만** 좋아진다 — `pr` 축은 사망(−0.0100·−0.0571), "
            "`p_miss` 는 반예측(−0.0205), `sc_dev ≡ p_score`. X6 상한은 그만큼 낙관적이다"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7z_direct_test", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q7-Z — 탈감쇠 모형 직접 검정** (Q7-Y 보다 먼저 돈다)")
run.log(f"  ★★★ 주 관문 Z1 — 적분 반폭 w∈{W_MS} ms · **w=0 이 `p_score`**")
run.log(f"       사전등록 예측: 초과비 w={W_KEY:.0f}/w=0 ≈ **{RATIO_PRED}**")
run.log(f"  ★★ 필요표본은 **두 값 병기** — null 옛 {REF['need']['null_old']} · "
        f"새 {REF['need']['null_new']} (기전 미규명)")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 【Z-0】 BUT PDB 적재 (Q7-X 와 동일 경로)
try:
    import wfdb, neurokit2 as nk
except ImportError:
    !pip -q install wfdb neurokit2
    importlib.invalidate_caches(); import wfdb, neurokit2 as nk
import re, urllib.request
run.log("\n" + "=" * 100)
run.log("【Z-0】 BUT PDB")
run.log("=" * 100)
BASE = f"https://physionet.org/files/{BUT_DIR}"
BEAT_SYM = set("NLRAaJSVFejE/fQ")

# ★★ PhysioNet 은 **일시적 502/503** 을 낸다(실측: `21.qrs` 에서 502 Bad Gateway).
#    재시도가 없으면 요청 하나가 다운로드 전체를 날린다. 그리고 더 나쁜 건
#    예외를 **삼키는** 코드다 — 일시 오류가 조용한 레코드 손실이 된다.
_TRANSIENT = ("502", "503", "504", "Bad Gateway", "Service Unavailable",
              "Timeout", "timed out", "Connection", "Temporary")

def _is_transient(e):
    m = f"{type(e).__name__}: {e}"
    return any(t in m for t in _TRANSIENT)

def net(fn, *a, **k):
    """일시 오류만 **지수 백오프로 재시도**한다. 영구 오류(404 등)는 즉시 올린다."""
    last = None
    for i in range(NET_TRIES):
        try:
            return fn(*a, **k)
        except Exception as e:
            last = e
            if not _is_transient(e) or i == NET_TRIES - 1:
                raise
            w = NET_BASE ** (i + 1)
            run.log(f"    ⏳ 일시 오류 재시도 {i+1}/{NET_TRIES-1} ({w:.0f}초) — "
                    f"{type(e).__name__}: {str(e)[:90]}")
            time.sleep(w)
    raise last

def _get(url, timeout=60):
    def _f():
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.read().decode("utf-8", "replace")
    return net(_f)

BUT_RECS = [str(r) for r in net(wfdb.get_record_list, "but-pdb")]
if len(BUT_RECS) < 10:
    raise AssetError(f"BUT PDB 목록이 {len(BUT_RECS)}개 — 다운로드 실패(R16)")

def resolve_rid(r):
    """★ RECORDS 는 `1` 인데 파일은 `01.hea` 다. 후보를 헤더로 확인한다.
    ⚠️ 일시 오류를 **「그 이름이 아니다」로 오해하면 안 된다** — `net()` 이 재시도하고,
      그래도 안 되면 **예외를 올려** 조용한 손실을 막는다."""
    cands = list(dict.fromkeys(
        [r] + ([f"{int(r):0{w}d}" for w in (2, 3)] if r.isdigit() else [])))
    for c in cands:
        try:
            net(wfdb.rdheader, c, pn_dir=BUT_DIR); return c
        except Exception as e:
            if _is_transient(e):
                raise                       # 재시도까지 실패한 일시 오류 → 삼키지 않는다
            continue                        # 진짜 「그 이름이 아니다」
    return None

_res, _net_fail = [], []
for r in BUT_RECS:
    try:
        _res.append(resolve_rid(r))
    except Exception as e:
        _net_fail.append((r, f"{type(e).__name__}: {str(e)[:80]}")); _res.append(None)
BUT_RECS = [c for c in _res if c is not None]
if _net_fail:
    run.log(f"  ⚠️ 이름 확인 중 **일시 오류로 못 읽은 레코드 {len(_net_fail)}개** — "
            f"{[r for r, _ in _net_fail][:8]}")

def list_exts(rid):
    exts = []
    try:
        for ln in _get(f"{BASE}/ANNOTATORS").splitlines():
            tok = ln.split("\t")[0].strip() if ln.strip() else ""
            if tok and not tok.startswith("#"):
                exts.append(tok.split()[0])
    except Exception as e:
        run.log(f"  ⚠️ ANNOTATORS 실패: {type(e).__name__}")
    if not exts:
        try:
            html = _get(BASE + "/")
            exts = sorted({x for x in re.findall(rf"{re.escape(rid)}\.([A-Za-z0-9_]+)", html)
                           if x not in ("dat", "hea", "xws", "png", "txt")})
        except Exception as e:
            run.log(f"  ⚠️ 디렉터리 목록 실패: {type(e).__name__}")
    return exts

PROBE = []
for e in list_exts(BUT_RECS[0]):
    try:
        a = net(wfdb.rdann, BUT_RECS[0], e, pn_dir=BUT_DIR)
        PROBE.append((e, len(a.sample), sorted(set(a.symbol))))
    except Exception:
        pass
if not PROBE:
    raise AssetError(f"{BUT_RECS[0]}: 읽히는 주석이 없다")
EXT_P = next((e for e, _, _ in PROBE if e.lower().startswith("p")), None)
_rest = [(e, n_, sy) for e, n_, sy in PROBE if e != EXT_P]
EXT_Q = max(_rest, key=lambda t: (len(set(t[2]) & BEAT_SYM), t[1]))[0] if _rest else None
if EXT_P is None or EXT_Q is None:
    raise AssetError(f"주석 역할을 못 가렸다 — {[(e, n) for e, n, _ in PROBE]}")

BUT, T0, SKIP = {}, time.time(), []
for rid in BUT_RECS:
    try:
        rec = net(wfdb.rdrecord, rid, pn_dir=BUT_DIR)
        rp = np.asarray(net(wfdb.rdann, rid, EXT_Q, pn_dir=BUT_DIR).sample, int)
        pp = np.asarray(net(wfdb.rdann, rid, EXT_P, pn_dir=BUT_DIR).sample, int)
    except Exception as e:
        SKIP.append((rid, f"{type(e).__name__}: {str(e)[:80]}")); continue
    sig = np.nan_to_num(np.asarray(rec.p_signal, float), nan=0.0, posinf=0.0, neginf=0.0)
    if len(rp) < 5 or len(pp) < MIN_P:
        continue
    BUT[rid] = dict(sig=sig[:, :2], fs=int(rec.fs), r=rp, p=pp)
if SKIP:
    run.log(f"  ⚠️ **재시도 후에도 못 받은 레코드 {len(SKIP)}개** — {[r for r, _ in SKIP][:8]}")
    for r_, m_ in SKIP[:3]:
        run.log(f"      {r_}: {m_}")
if len(BUT) < REC_FLOOR:
    raise AssetError(
        f"적재 레코드가 {len(BUT)}개로 바닥 {REC_FLOOR} 미만이다(기대 {REC_EXPECT}). "
        f"일시 오류 {len(_net_fail)+len(SKIP)}건 — PhysioNet 이 불안정하면 **잠시 뒤 "
        "다시 돌려라**. 레코드가 줄면 λ 가 앞선 런과 비교 불가해진다(R16)")
if len(BUT) != REC_EXPECT:
    run.log(f"  ⚠️ **적재 {len(BUT)}개 ≠ 기대 {REC_EXPECT}개** — Q7-U·V·X 와 코호트가 "
            "다르므로 λ 를 그 런들과 나란히 놓을 때 이 차이를 적어라")
FSSET = sorted({v["fs"] for v in BUT.values()})
run.log(f"  적재 {len(BUT)}/{len(BUT_RECS)}개 · {time.time()-T0:.0f}초 · 표본율 {FSSET}Hz (기대 {REC_EXPECT})")
run.log(f"  ★ 1샘플 = " + " · ".join(f"{f}Hz→{1000.0/f:.2f}ms" for f in FSSET))
run.log("  ⚠️ 그래서 Δ 를 **ms 가 아니라 샘플**로 잡는다 — 128Hz 에서 Δ=5ms 와 11ms 가")
run.log("     둘 다 1샘플로 반올림돼 X1 의 λ(11ms) 가 **위로 편향**됐다(외부 검토 지적)")
CONFIG["but"] = dict(n_rec=len(BUT), fs=[int(f) for f in FSSET],
                     expect=REC_EXPECT, skipped=[r for r, _ in SKIP],
                     name_fail=[r for r, _ in _net_fail])
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【Z-A】 Z3 — λ 를 **샘플 단위 · 표본율 층화 · w 스윕**으로 다시
run.log("\n" + "=" * 100)
run.log("【Z-A】 Z3 — λ(Δ 샘플) × 적분반폭 w × 표본율 층")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def ms2s(ms, fs):
    return int(round(ms * fs / 1000.0))

def _detrend(v):
    n = len(v)
    if n < 3:
        return np.asarray(v, float)
    t = np.arange(n, dtype=float)
    a, b = np.polyfit(t, np.asarray(v, float), 1)
    return np.asarray(v, float) - (a * t + b)

def _win(x, q, fs):
    """공통 창·분모 — 전 w 가 **같은 것**을 쓴다. w 축 하나만 바뀐다(R34 ③)."""
    w = ms2s(SCORE_HALF_MS, fs)
    a, b = max(int(q) - w, 0), min(int(q) + w + 1, len(x))
    if b - a < 5:
        return None, None, None
    seg = _detrend(x[a:b])
    den = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
    return seg, int(q) - a, den

def score_w(x, pos, fs, w_ms):
    """★ `w_ms=0` 이면 **분자가 단일 표본** = `p_score`. 그 외는 ±w_ms RMS."""
    h = ms2s(w_ms, fs)
    out = []
    for q in np.atleast_1d(pos):
        seg, c, den = _win(x, q, fs)
        if seg is None:
            out.append(0.0); continue
        if h <= 0:
            out.append(float(abs(seg[c]) / den)); continue
        a, b = max(c - h, 0), min(c + h + 1, len(seg))
        out.append(float(np.sqrt(np.mean(seg[a:b] ** 2)) / den))
    return np.asarray(out, float)

def true_p_positions(rid):
    d = BUT[rid]; fs = d["fs"]
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    out = []
    for pt in d["p"]:
        nxt = d["r"][d["r"] > pt]
        if not len(nxt):
            continue
        R = int(nxt[0])
        if R + lo <= pt < R + hi:
            out.append(int(pt))
    return np.asarray(out, int)

# ── ⓐ 고정 Δ(샘플) × w — 표본율 층별
T1_ = time.time()
LAMS = {}                       # (fs, w, dsamp) -> [레코드별 λ]
for rid in sorted(BUT):
    d = BUT[rid]; fs = d["fs"]; x = d["sig"][:, 0]
    pt = true_p_positions(rid)
    if len(pt) < 20:
        continue
    for w in W_MS:
        v0 = score_w(x, pt, fs, w)
        if np.std(v0) < 1e-12:
            continue
        for ds in D_SAMP:
            if ds == 0:
                LAMS.setdefault((fs, w, ds), []).append(1.0)   # ★ 구성상 항등
                continue
            cs = []
            for sgn in (+1, -1):
                v1 = score_w(x, np.clip(pt + sgn * ds, 0, len(x) - 1), fs, w)
                if np.std(v1) > 1e-12:
                    cs.append(float(np.corrcoef(v0, v1)[0, 1]))
            if cs:
                LAMS.setdefault((fs, w, ds), []).append(float(np.mean(cs)))
run.log(f"  ({time.time()-T1_:.0f}초) 층별 λ 계산 완료")

for f_ in FSSET:
    z = LAMS.get((f_, 0.0, 0), [])
    if z and float(np.max(np.abs(np.asarray(z) - 1.0))) > 1e-12:
        raise AssetError(f"{f_}Hz: λ(Δ=0) 이 1.0 이 아니다 — 곡선 전체가 무의미하다")
run.log("  ★ 항등 대조 — λ(Δ=0 샘플) = 1.000000 (전 층·전 w · 구성으로 보장)")

Z3 = {}
for f_ in FSSET:
    n_rec = len(LAMS.get((f_, 0.0, D_KEY_S), []))
    if n_rec < 5:
        run.log(f"\n  {f_}Hz — 레코드 {n_rec}개, 층 판정 생략(R17)")
        continue
    run.log(f"\n  ▸ **{f_}Hz 층** (1샘플 = {1000.0/f_:.2f}ms · 레코드 {n_rec})")
    run.log(f"    {'Δ(샘플)':>8}{'=ms':>8}" + "".join(f"{('w='+str(int(w))):>9}" for w in W_MS))
    for ds in D_SAMP:
        row = []
        for w in W_MS:
            m_, _, _, _ = boot_mean(LAMS.get((f_, w, ds), []), SEED0 + 11)
            row.append(m_)
            Z3[f"{f_}|{w}|{ds}"] = float(m_)
        run.log(f"    {ds:>8}{ds*1000.0/f_:>8.1f}" + "".join(f"{v:>9.4f}" for v in row))

# ── ⓑ ★ 실측 오차 분포 하 λ_w — X1 의 고정-Δ λ_energy 에 **짝을 맞춘다**
run.log("\n  ★ Z3b — **실측 검출기 오차 분포** 하 λ_w (X1 은 고정 Δ 였다 · 검토 지적 ⑤)")
def detect_dwt(x, rp, fs):
    _, w = nk.ecg_delineate(x, rpeaks=rp, sampling_rate=fs, method=DELIN)
    p = w.get("ECG_P_Peaks", [])
    return np.asarray(sorted({int(v) for v in p
                              if v is not None and np.isfinite(v) and 0 <= v < len(x)}), int)

T2_ = time.time()
LAM_REAL = {w: [] for w in W_MS}
LAM_REAL_FS = {(f_, w): [] for f_ in FSSET for w in W_MS}
for rid in sorted(BUT):
    d = BUT[rid]; fs = d["fs"]; x = d["sig"][:, 0]
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    try:
        pos = detect_dwt(x, d["r"], fs)
    except Exception:
        continue
    tp, op = [], []
    for pt in d["p"]:
        nxt = d["r"][d["r"] > pt]
        if not len(nxt):
            continue
        R = int(nxt[0])
        if not (R + lo <= pt < R + hi):
            continue
        cand = pos[(pos >= R + lo) & (pos < R + hi)]
        if not len(cand):
            continue
        tp.append(int(pt)); op.append(int(cand[np.argmin(np.abs(cand - (R + lo + hi) // 2))]))
    if len(tp) < 20:
        continue
    tp = np.asarray(tp, int); op = np.asarray(op, int)
    for w in W_MS:
        a_ = score_w(x, tp, fs, w); b_ = score_w(x, op, fs, w)
        if np.std(a_) > 1e-12 and np.std(b_) > 1e-12:
            c_ = float(np.corrcoef(a_, b_)[0, 1])
            LAM_REAL[w].append(c_); LAM_REAL_FS[(fs, w)].append(c_)
run.log(f"    ({time.time()-T2_:.0f}초) {'w(ms)':>7}{'λ(전체)':>10}{'CI':>22}"
        + "".join(f"{(str(f)+'Hz'):>10}" for f in FSSET))
Z3B = {}
for w in W_MS:
    m_, lo_, hi_, n_ = boot_mean(LAM_REAL[w], SEED0 + 21 + int(w))
    per = [float(np.nanmean(LAM_REAL_FS[(f_, w)])) if LAM_REAL_FS[(f_, w)] else np.nan
           for f_ in FSSET]
    Z3B[w] = dict(lam=m_, lo=lo_, hi=hi_, n=n_,
                  by_fs={int(f_): float(v) for f_, v in zip(FSSET, per)})
    run.log(f"    {w:>7.0f}{m_:>10.4f}  [{lo_:>7.4f},{hi_:>7.4f}]"
            + "".join(f"{v:>10.4f}" for v in per))
best_w = max(W_MS, key=lambda w: Z3B[w]["lam"])
lam0, lamb = Z3B[0.0]["lam"], Z3B[best_w]["lam"]
LAM_128 = Z3B[W_KEY]["by_fs"].get(128, float("nan"))
run.log(f"    ★ 실측 오차 하 최량 w = **{best_w:.0f}ms** (λ {lamb:.4f} vs w=0 {lam0:.4f})")
run.log(f"    ★ **SVDB 는 128Hz** — X6 계수로 쓸 값은 128Hz 층의 λ 다 "
        f"(w={W_KEY:.0f} 에서 {LAM_128:.4f})")
g_("Z3", "✅ 지지" if np.isfinite(lamb) and lamb > lam0 else "❌ 기각",
   f"실측 오차 분포 하에서도 적분이 λ 를 올린다 — w=0 {lam0:.4f} → w={best_w:.0f} "
   f"**{lamb:.4f}** (성분 병기 · R36 ⑤)" if lamb > lam0 else
   f"★ 실측 오차 분포 하에서는 적분이 λ 를 **못 올린다**(w=0 {lam0:.4f} · 최량 {lamb:.4f}) — "
   "X1 의 고정-Δ 이득은 총오검출 꼬리에서 사라진다")
CONFIG["Z3"] = Z3; CONFIG["Z3b"] = {str(k): v for k, v in Z3B.items()}
CONFIG["best_w"] = float(best_w); CONFIG["lam_128"] = float(LAM_128)
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【Z-B】 SVDB 특징 + ★ Z0 좌표 이해 증명
import pandas as pd
run.log("\n" + "=" * 100)
run.log("【Z-B】 SVDB 특징 · Z0 — 비트 배열 재계산이 자산과 얼마나 다른가")
run.log("=" * 100)
for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "Q7-P0")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True); PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); SYM = np.asarray(D5["sym"]).astype(str)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
if int((np.asarray(PD["pid"]).astype(int) != PID).sum()) or \
   int((np.asarray(PD["sym"]).astype(str) != SYM).sum()):
    raise AssetError("정합 깨짐 — Q7-P0 를 다시 돌린다")
P_IDX = np.asarray(PD["p_idx"]).astype(int); P_SC = np.asarray(PD["p_score"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx0 = P_IDX[K].copy(); psc_asset = P_SC[K].copy()
RS = np.array(sorted(set(RID.tolist())))
XB = np.ascontiguousarray(np.asarray(D5["beat"])[K][:, 0, :]).astype(float)

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))

# ── ★ 비트 배열 점수 — **전 w 가 같은 창·같은 분모**(w 축 하나만)
HW = int(round(SCORE_HALF_MS * FS / 1000.0))
_OFFD = np.arange(-HW, HW + 1)
_Tc = _OFFD.astype(float) - _OFFD.mean(); _Tss = float((_Tc ** 2).sum())

def beat_scores(pos, rows, w_ms):
    """비트 배열에서 계산. ★ 창이 배열 밖으로 나가면 **Q7-P0 와 같은 규약으로 절단**한다
    (그래서 자산과 corr 0.71 — 연속 신호에는 있던 왼쪽 문맥이 여기 없다)."""
    X = XB[rows]
    p = np.clip(np.asarray(pos, int), 0, X.shape[1] - 1)
    h = int(round(w_ms * FS / 1000.0))
    out = np.zeros(len(p))
    for i in range(len(p)):
        q = int(p[i])
        a, b = max(q - HW, 0), min(q + HW + 1, X.shape[1])
        seg = X[i, a:b]
        if len(seg) < 5:
            continue
        t = np.arange(len(seg), dtype=float); tc = t - t.mean()
        sl = float((seg * tc).sum() / max((tc ** 2).sum(), 1e-12))
        seg = seg - (sl * tc + seg.mean())
        den = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
        c = q - a
        if h <= 0:
            out[i] = abs(seg[c]) / den
        else:
            u, v = max(c - h, 0), min(c + h + 1, len(seg))
            out[i] = float(np.sqrt(np.mean(seg[u:v] ** 2)) / den)
    return out

FIRE = pidx0 >= 0
T3_ = time.time()
SC = {}
for w in W_MS:
    s = np.zeros(len(K))
    s[FIRE] = beat_scores(pidx0[FIRE], FIRE, w)
    SC[w] = s                                # 미발화는 0 (Q7-P0 규약)
run.log(f"  ({time.time()-T3_:.0f}초) 비트 {len(K):,} × w {len(W_MS)}개 계산 완료")

# ── ★★ Z0 — **구성적 항등 증명** (숫자 재현이 아니다)
#    창이 비트 배열 안에 **온전히 들어가는** 비트(`HW ≤ p_idx ≤ L-HW-1`)에서는
#    비트 창이 연속 신호 창과 **정확히 같은 표본**을 담는다. 거기서 corr≈1 이면
#    「우리 계산이 Q7-P0 의 계산이다」가 **구성으로** 증명되고, 불일치가 **잘린
#    비트에만** 있다는 것도 같이 증명된다.
#    ⚠️ Q7-V 1판의 0.7122 를 기준으로 쓰지 않는다 — 그건 창 중심을 옮기던 **버그**
#      코드의 출력이다. **버그 있는 코드의 출력에 재현 기준을 앵커하면 안 된다.**
FIT = FIRE & (pidx0 >= HW) & (pidx0 <= XB.shape[1] - HW - 1)
n_fit, n_cut = int(FIT.sum()), int((FIRE & ~FIT).sum())
if n_fit < 1000:
    raise AssetError(f"창이 온전한 비트가 {n_fit:,}개뿐 — 항등 증명을 세울 수 없다")
c_fit = float(np.corrcoef(SC[0.0][FIT], psc_asset[FIT])[0, 1])
d_fit = float(np.median(np.abs(SC[0.0][FIT] - psc_asset[FIT])))
c_all = float(np.corrcoef(SC[0.0][FIRE], psc_asset[FIRE])[0, 1])
d_all = float(np.median(np.abs(SC[0.0][FIRE] - psc_asset[FIRE])))
run.log(f"\n  ★★ Z0 — **구성적 항등 증명**  (창 반폭 {HW}샘플 = {SCORE_HALF_MS:.0f}ms)")
run.log(f"    창이 **온전한** 비트  {n_fit:>7,} ({n_fit/max(int(FIRE.sum()),1):.1%}) — "
        f"corr **{c_fit:.6f}** · |Δ| 중앙 {d_fit:.6f}")
run.log(f"    창이 **잘린** 비트    {n_cut:>7,} ({n_cut/max(int(FIRE.sum()),1):.1%}) — "
        "연속 신호에는 있던 **왼쪽 문맥이 비트 배열엔 없다**")
run.log(f"    전체                       corr {c_all:.4f} · |Δ| 중앙 {d_all:.6f}")
run.log(f"    (참고 · Q7-V 1판 0.{int(REF['asset_corr_v1bug']*10000):04d} 는 창 중심을 "
        "`np.clip` 으로 **옮기던 버그** 코드의 값이다 — 기준으로 쓰지 않는다)")
ok0 = (c_fit >= FIT_CORR_MIN) and (d_fit <= FIT_MED_MAX) and (c_fit > c_all)
g_("Z0", "✅ 지지" if ok0 else "❌ 기각",
   (f"★ 창이 온전한 비트에서 corr **{c_fit:.6f}** ≥ {FIT_CORR_MIN} 이고 전체({c_all:.4f})"
    "보다 높다 — **우리 계산 = Q7-P0 계산**이고 불일치는 **잘린 비트에만** 있다"
    if ok0 else
    f"★ 항등이 안 선다 — 온전한 비트 corr {c_fit:.6f}(요구 ≥{FIT_CORR_MIN}) · "
    f"|Δ| 중앙 {d_fit:.6f}(요구 ≤{FIT_MED_MAX}) · 전체 {c_all:.4f}"))
if not ok0:
    raise AssetError(
        f"Z0 실패 — 창이 **온전히 들어가는** 비트에서조차 자산과 일치하지 않는다"
        f"(corr {c_fit:.6f} · |Δ| 중앙 {d_fit:.6f}). 그 비트들은 연속 신호와 **같은 "
        "표본**을 담으므로 반드시 일치해야 한다. 좌표나 창 규약이 어긋난 것이고, "
        "그 상태로는 w 스윕이 무엇을 재는지 알 수 없다")
run.log("    ▸ **w 간 비교는 같은 계산 안에서** 이뤄지므로, 잘린 비트가 있어도 유효하다 —")
run.log("      전 w 가 같은 창·같은 분모·같은 절단 규약을 쓴다(축이 하나뿐)")
CONFIG["Z0"] = dict(corr_fit=c_fit, med_fit=d_fit, corr_all=c_all,
                    med_all=d_all, n_fit=n_fit, n_cut=n_cut, ok=bool(ok0))
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【Z-C】 ★★★ Z1 주 관문 — w 스윕으로 S3 재측정 · **비 예측 검정**
run.log("\n" + "=" * 100)
run.log("【Z-C】 Z1(주) — 적분 반폭 w 스윕 · 사전등록 예측 비 " + str(RATIO_PRED))
run.log("=" * 100)
run.log("  ▸ **w=0 이 곧 `p_score`** 다(분자 단일 표본). 분모·창은 전 w 동일 —")
run.log("    바뀌는 축이 **하나뿐**이라 비교가 유효하다(R34 ③)")

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    X = basis_ext(idx); y = v[idx]
    okm = np.isfinite(y)
    if okm.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[okm], y[okm], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx, perm=None, rng=None):
    tt = TT[idx]; key = np.round(f1[idx]).astype(int)
    if perm == "record":
        tt = tt.copy()
        for u in np.unique(RID[idx]):
            m = np.where(RID[idx] == u)[0]
            tt[m] = tt[m][rng.permutation(len(m))]
    elif perm == "stratum":
        tt = tt.copy()
        for kk in np.unique(key):
            m = np.where(key == kk)[0]
            tt[m] = tt[m][rng.permutation(len(m))]
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

REC_OK = [r for r in RS
          if (lambda i: TT[i].sum() >= MIN_S and (~TT[i]).sum() >= MIN_N)(np.where(RID == r)[0])]
IDXS = {r: np.where(RID == r)[0] for r in REC_OK}

T4_ = time.time()
PERW, Z1 = {}, {}
run.log(f"\n  {'w(ms)':>7}{'AUROC':>10}{'CI':>22}{'초과(0.5)':>11}{'n':>5}{'쌍중앙':>9}")
for w in W_MS:
    per, npr, rr = [], [], []
    for r in REC_OK:
        idx = IDXS[r]
        a, np_ = matched_auc(resid(SC[w], idx), idx)
        if np.isfinite(a):
            per.append(a); npr.append(np_); rr.append(r)
    m_, lo_, hi_, n_ = boot_mean(per, SEED0 + 51)
    PERW[w] = dict(per=per, rid=rr, npair=npr)
    Z1[w] = dict(auc=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)),
                 excess=float(m_ - 0.5), pairs=int(np.median(npr)) if npr else 0)
    run.log(f"  {w:>7.0f}{m_:>10.4f}  [{lo_:>7.4f},{hi_:>7.4f}]{m_-0.5:>+11.4f}{n_:>5}"
            f"{Z1[w]['pairs']:>9,}")
run.log(f"  ({time.time()-T4_:.0f}초)")

# ── ★★★ 예측 검정 — **짝지은 차**(주) + **비**(예측 대조)
base = np.asarray(PERW[0.0]["per"], float)
key_ = np.asarray(PERW[W_KEY]["per"], float)
same_rec = PERW[0.0]["rid"] == PERW[W_KEY]["rid"]
if not same_rec:
    run.log("  ⚠️ w=0 과 w=key 의 판정 레코드 집합이 다르다 — 교집합으로 짝을 맞춘다")
    common = [r for r in PERW[0.0]["rid"] if r in set(PERW[W_KEY]["rid"])]
    base = np.array([PERW[0.0]["per"][PERW[0.0]["rid"].index(r)] for r in common])
    key_ = np.array([PERW[W_KEY]["per"][PERW[W_KEY]["rid"].index(r)] for r in common])

# ★ 주 통계량은 **차**다 — 비는 분모(w=0 초과)가 0 근처면 폭발한다
dm, dlo, dhi, dn = boot_pair(base, key_, lambda a, b: float(b.mean() - a.mean()),
                             SEED0 + 60)
def _ratio(a, b):
    ea, eb = a.mean() - 0.5, b.mean() - 0.5
    return float(eb / ea) if abs(ea) > 1e-4 else float("nan")
rm, rlo, rhi, rn = boot_pair(base, key_, _ratio, SEED0 + 61)
e0, ek = Z1[0.0]["excess"], Z1[W_KEY]["excess"]

# ★ 실측 오차·**128Hz 층** 기반 수정 예측 (SVDB 가 128Hz 이므로 이쪽이 옳은 눈금)
l0_128 = Z3B[0.0]["by_fs"].get(128, float("nan"))
lk_128 = Z3B[W_KEY]["by_fs"].get(128, float("nan"))
PRED_128 = float(lk_128 / l0_128) if np.isfinite(l0_128) and l0_128 > 0.05 else float("nan")
run.log(f"\n  ★★★ 사전등록 예측 검정 — `w={W_KEY:.0f}` vs `w=0`")
run.log(f"    성분(R36 ⑤) — w=0 초과 **{e0:+.4f}** · w={W_KEY:.0f} 초과 **{ek:+.4f}**")
run.log(f"    ★ **차(주)** {dm:+.4f} [{dlo:+.4f}, {dhi:+.4f}] · n={dn}")
run.log(f"      비(참고) {rm:.3f} [{rlo:.3f}, {rhi:.3f}]")
run.log(f"    예측 — 사전등록 **{RATIO_PRED}**(X1 고정 Δ) · "
        f"수정 **{PRED_128:.3f}**(Z3b 실측 오차 · **128Hz 층** — SVDB 눈금)")
run.log("    ▸ 주 통계량을 **차**로 둔 이유: 비는 분모(w=0 초과)가 0 근처면 폭발한다")
# ★★ 예측 비교도 **차**로 한다 — 비 CI 는 폭이 넓어 **무엇도 배제하지 못한다**
#    (1판은 관문만 차로 하고 예측 비교를 비로 남겨, 배제되는 예측을 「CI 안」으로 찍었다)
PD_PRE = e0 * (RATIO_PRED - 1.0)
PD_128 = e0 * (PRED_128 - 1.0) if np.isfinite(PRED_128) else float("nan")
run.log(f"    ★★ **예측을 차로 환산** — 사전등록 {RATIO_PRED} → **{PD_PRE:+.4f}** · "
        f"수정 {PRED_128:.3f} → **{PD_128:+.4f}**  (관측 차 CI [{dlo:+.4f}, {dhi:+.4f}])")
excl_pre = np.isfinite(dhi) and PD_PRE > dhi
excl_128 = np.isfinite(dhi) and np.isfinite(PD_128) and PD_128 > dhi
for nm_, pv_, ex_ in (("사전등록", PD_PRE, excl_pre), ("수정(128Hz)", PD_128, excl_128)):
    run.log(f"      {nm_:<12} 예측 차 {pv_:+.4f} → "
            + ("**CI 밖 = 배제**" if ex_ else "CI 안 = 배제 못 함"))
if not np.isfinite(dm) or dn < 5:
    g_("Z1", "⛔ 측정 불가", "짝지은 레코드가 5 미만")
elif excl_pre and excl_128:
    g_("Z1", "❌ 기각",
       f"★★★ **탈감쇠 모형 기각** — 관측 차 {dm:+.4f} [{dlo:+.4f}, {dhi:+.4f}] 가 "
       f"**두 예측 모두**({PD_PRE:+.4f} · {PD_128:+.4f})를 배제한다. λ 는 "
       f"{Z3B[0.0]['lam']:.4f}→{Z3B[W_KEY]['lam']:.4f} 로 올랐는데 **효용은 안 움직였다**")
    run.log("       ▸ ★ **λ 는 신뢰도(reliability)를 재지 타당도(validity)를 재지 않는다.**")
    run.log("         탈감쇠는 「정답 위치 값이 판별 정보를 담는다」를 가정하는데 그게 거짓이다")
    run.log("       ▸ → X6 의 상한 **전부 무효** · 「필요표본 1/4」 **폐기** ·")
    run.log("         **「자가 병목」 결론 철회**(근거였던 V1+X1 의 사슬이 여기서 끊긴다)")
    run.log("       ▸ → 결정 숫자는 **원 W1** 이다 — 어떤 자 개선으로도 부풀릴 수 없다")
else:
    v1_ = decide(dlo, dhi, 0.0, ">")
    g_("Z1", v1_,
       f"★★★ **차 {dm:+.4f}** [{dlo:+.4f}, {dhi:+.4f}] · 예측 차 사전등록 {PD_PRE:+.4f}"
       f"{'(배제)' if excl_pre else ''} · 수정 {PD_128:+.4f}{'(배제)' if excl_128 else ''}")
    if v1_.startswith("✅"):
        run.log("       ▸ **적분이 효용을 올린다** — λ 이득이 전환된다")
    else:
        run.log("       ▸ 이 표본에서 확인되지 않는다 — X6 을 **「미검정」**으로 표시한다")
DIFF["Z1"] = dict(diff=dm, dlo=dlo, dhi=dhi, ratio=rm, lo=rlo, hi=rhi,
                  e0=e0, ek=ek, pred=RATIO_PRED, pred_128=PRED_128,
                  pred_diff=float(PD_PRE), pred_diff_128=float(PD_128),
                  excl_pre=bool(excl_pre), excl_128=bool(excl_128), n=int(dn))
BEST_S3 = max(W_MS, key=lambda w: Z1[w]["excess"])
run.log(f"    ▸ 관측상 최량 w = {BEST_S3:.0f}ms (초과 {Z1[BEST_S3]['excess']:+.4f}) — "
        "★ **성적표에서 고른 값이라 다음 런에서 다시 검정해야 한다**(R34 ②)")
CONFIG["Z1"] = {str(k): {kk: vv for kk, vv in v.items()} for k, v in Z1.items()}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【Z-D】 Z2 null 기전 · Z4 쌍 수 ρ 영점
run.log("\n" + "=" * 100)
run.log("【Z-D】 Z2 — null 기전(`reps` 사다리 + nan 드롭) · Z4 — ρ(쌍 수) 영점")
run.log("=" * 100)
run.log("  ▸ 「MC 잡음」설명은 **틀렸다** — 중심을 안 옮기고 CI 를 넓힌다.")
run.log("    후보: `matched_auc` 가 `tot<MIN_PAIR` 에서 nan 을 내는데 치환하면")
run.log("    `tot = Σ n_S(k)·n_N(k)` 가 **바뀌고**, `nanmean` 이 살아남은 것만 평균하면")
run.log("    **선택 효과**가 생긴다. 드롭 수를 같이 찍어 가른다")

T5_ = time.time()
Z2 = {}
run.log(f"\n  {'reps':>6}{'레코드 안':>11}{'CI':>22}{'쌍 내':>10}{'CI':>22}{'nan 드롭':>10}")
for reps in REPS:
    vals = {"record": [], "stratum": []}
    drops = 0
    for r in REC_OK:
        idx = IDXS[r]
        vr = resid(SC[0.0], idx)
        for mode in vals:
            got = []
            for s in range(reps):
                rng = np.random.RandomState(SEED0 + 700 + s)
                a, _ = matched_auc(vr, idx, perm=mode, rng=rng)
                if np.isfinite(a):
                    got.append(a)
                else:
                    drops += 1
            if got:
                vals[mode].append(float(np.mean(got)))
    rowm = {}
    for mode in vals:
        m_, lo_, hi_, n_ = boot_mean(vals[mode], SEED0 + 71 + reps)
        rowm[mode] = dict(null=m_, lo=lo_, hi=hi_, n=n_)
    Z2[reps] = dict(**rowm, drops=int(drops))
    run.log(f"  {reps:>6}{rowm['record']['null']:>11.4f}"
            f"  [{rowm['record']['lo']:>7.4f},{rowm['record']['hi']:>7.4f}]"
            f"{rowm['stratum']['null']:>10.4f}"
            f"  [{rowm['stratum']['lo']:>7.4f},{rowm['stratum']['hi']:>7.4f}]{drops:>10,}")
run.log(f"  ({time.time()-T5_:.0f}초)")
n3, n50 = Z2[REPS[0]]["record"]["null"], Z2[REPS[-1]]["record"]["null"]
conv = abs(n50 - 0.5) < abs(n3 - 0.5) and abs(n3 - 0.5) > 0.005
g_("Z2", "✅ 지지" if conv else "⚠️ 미결",
   (f"★ `reps` 를 늘리면 null 이 0.5 로 **수렴**한다({n3:.4f} → {n50:.4f}) — "
    "Q7-S′ 의 0.5090 은 소수 치환의 산물이다" if conv else
    f"reps 를 늘려도 null 이 거의 안 움직인다({n3:.4f} → {n50:.4f}) — "
    "**Q7-S′ 의 0.5090 기전은 여전히 미규명**이다. 필요표본을 **두 값 병기**로 유지한다"))
run.log(f"    ▸ 필요표본(우월) — null 옛 기준 **{REF['need']['null_old']}** · "
        f"새 기준 **{REF['need']['null_new']}** (50%/80%) — **확정 전까지 병기**(R30 ①)")

# ── Z4 — ρ(쌍 수) 영점
run.log("\n  ★ Z4 — `ρ(쌍 수)` 에 라벨셔플 영점")
per0 = np.asarray(PERW[0.0]["per"], float)
npr0 = np.log1p(np.asarray(PERW[0.0]["npair"], float))
rho_obs = spearman(npr0, per0)
rho_nul = []
for s in range(30):
    rng = np.random.RandomState(SEED0 + 800 + s)
    pn = []
    for r, idx in ((r, IDXS[r]) for r in PERW[0.0]["rid"]):
        a, _ = matched_auc(resid(SC[0.0], idx), idx, perm="stratum", rng=rng)
        pn.append(a)
    pn = np.asarray(pn, float)
    if np.isfinite(pn).sum() >= 5:
        rho_nul.append(spearman(npr0[np.isfinite(pn)], pn[np.isfinite(pn)]))
nm_, nlo, nhi, _ = boot_mean(rho_nul, SEED0 + 81)
run.log(f"    관측 ρ **{rho_obs:+.4f}** (Q7-X {REF['rho_pairs']:+.4f}) · "
        f"영점 ρ {nm_:+.4f} [{nlo:+.4f}, {nhi:+.4f}]")
g_("Z4", "✅ 지지" if np.isfinite(nhi) and rho_obs > nhi else "⚠️ 미결",
   ("쌍 수 연관이 **영점을 넘는다** — 인공물이 아니다" if np.isfinite(nhi) and rho_obs > nhi
    else "쌍 수 연관이 영점 범위 안이다 — **인공물과 구분되지 않는다**"))
CONFIG["Z2"] = {str(k): v for k, v in Z2.items()}
CONFIG["Z4"] = dict(rho=float(rho_obs), null=float(nm_), null_lo=float(nlo),
                    null_hi=float(nhi))
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【Z-E】 ★ Z5 결론 검산표 (R38 ①) — 코드에 고정한다
run.log("\n" + "=" * 100)
run.log("【Z-E】 Z5 — **결론 검산표**: 근거 숫자 · 미검정 가정 · 가정이 틀리면")
run.log("=" * 100)
run.log("  ▸ 지금까지 이 검산을 **대화로** 해왔다. 코드에 넣으면 매번 자동으로 걸린다")

lam_x6 = LAM_128 if np.isfinite(LAM_128) else Z3B[W_KEY]["lam"]
w1 = REF["w1"]
def _de(v, l):
    return float(v / max(min(l, 1.0), 1e-6)) if np.isfinite(l) and l > 0.05 else float("nan")

CHECK = [
    dict(claim="자가 병목이다 (분자가 점이라 위치에 과민)",
         num=f"λ(Δ={D_KEY_S}샘플, w=0) 층별 · V1 실측 λ {REF['lam_v']:.4f}",
         assume="**없음** — X1 은 정답 위치만 흔들었고 Δ=0 항등이 붙었다",
         iffalse="—"),
    dict(claim=f"적분이 λ 를 올린다 (w=0 {Z3B[0.0]['lam']:.4f} → "
               f"w={best_w:.0f} {Z3B[best_w]['lam']:.4f})",
         num="Z3b — **실측 오차 분포** 하에서 잰 값(X1 은 고정 Δ 였다)",
         assume="BUT PDB → SVDB **전이 가정**(코호트·표본율)",
         iffalse="SVDB 는 128Hz 로 더 거칠어 실제 λ 는 **더 낮다** → 이득도 더 작다"),
    dict(claim=f"λ 이득이 **효용**으로 전환된다 (차 {DIFF['Z1']['diff']:+.4f})",
         num=f"Z1 — w=0 초과 {DIFF['Z1']['e0']:+.4f} vs w={W_KEY:.0f} {DIFF['Z1']['ek']:+.4f}" + f" · Z0 항등 corr {CONFIG['Z0']['corr_fit']:.6f}",
         assume="**직접 측정이라 모형 가정 없음** — 이게 이 런의 요점이다",
         iffalse="—"),
    dict(claim="탈감쇠 상한 W1 ≈ " + f"{_de(w1['hi'], lam_x6):+.4f}",
         num=f"W1 상한 {w1['hi']:+.4f} ÷ λ {lam_x6:.4f}",
         assume="① 선형성·오차독립 ② 이분정규(AUROC↔d) ③ **W1 5차원 중 1축만** 좋아짐",
         iffalse="③ 때문에 이미 **낙관적**이다 — `pr` 축 사망 · `p_miss` 반예측 · "
                 "`sc_dev≡p_score`"),
    dict(claim=f"필요표본 (우월 50%/80%)",
         num=f"null 옛 {REF['need']['null_old']} · 새 {REF['need']['null_new']}",
         assume=f"null 기전 미규명 — Z2 판정 {VERD.get('Z2', '?')}",
         iffalse="**두 값을 병기한다.** 하나만 실으면 그게 사실로 굳는다"),
    dict(claim="W1 점추정 상당",
         num=f"{_de(w1['mean'], lam_x6):+.4f} "
             f"[{_de(w1['lo'], lam_x6):+.4f}, {_de(w1['hi'], lam_x6):+.4f}]",
         assume="관측 W1 자체가 **0 을 덮는다**",
         iffalse="★ **CI 없이 인용 금지** — 「+4%」로 굳으면 안 된다(R36 ①)"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["Z5"] = CHECK
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【Z-F】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① ★★★ Z1 — w 스윕 초과 (주 관문)
ws = list(W_MS)
ex = [Z1[w]["excess"] for w in ws]
lo = [Z1[w]["lo"] - 0.5 for w in ws]; hi = [Z1[w]["hi"] - 0.5 for w in ws]
ax[0].plot(ws, ex, "o-", color="tab:red", label="S3 excess over 0.5")
ax[0].fill_between(ws, lo, hi, color="tab:red", alpha=.15)
ax[0].axhline(0, color="k", lw=.9)
ax[0].axhline(ex[0], ls=":", color="tab:gray", lw=1.0)
ax[0].annotate("w=0 (p_score)", (ws[-1], ex[0]), fontsize=7, ha="right", va="bottom")
for pv, cl, tag in ((RATIO_PRED, "tab:blue", "prereg"),
                    (DIFF['Z1']['pred_128'], "tab:green", "128Hz revised")):
    if np.isfinite(pv):
        ax[0].axhline(ex[0] * pv, ls="--", color=cl, lw=1.0)
        ax[0].annotate(f"{tag} x{pv:.2f}", (ws[-1], ex[0] * pv),
                       fontsize=7, ha="right", va="bottom", color=cl)
ax[0].set_xlabel("numerator integration half-width w (ms)")
ax[0].set_ylabel("S3 excess")
ax[0].set_title(f"Z1 : diff {DIFF['Z1']['diff']:+.4f} · ratio "
                f"{DIFF['Z1']['ratio']:.2f}", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② Z3b — 실측 오차 하 λ_w (표본율 층별)
for f_, c in zip(FSSET, ("tab:blue", "tab:orange", "tab:green")):
    y = [Z3B[w]["by_fs"].get(int(f_), np.nan) for w in ws]
    ax[1].plot(ws, y, "s-", color=c, label=f"{f_} Hz")
ax[1].plot(ws, [Z3B[w]["lam"] for w in ws], "o--", color="k", label="pooled")
ax[1].axhline(REF["lam_v"], ls=":", color="tab:gray", lw=1.0)
ax[1].annotate(f"V1 lambda={REF['lam_v']:.3f}", (ws[-1], REF["lam_v"]),
               fontsize=7, ha="right", va="bottom")
ax[1].set_xlabel("numerator integration half-width w (ms)")
ax[1].set_ylabel("lambda under MEASURED detector error")
ax[1].set_title("Z3b : does integration survive the real error mix?", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

# ③ Z2 — null 사다리
rp = list(REPS)
ax[2].errorbar(rp, [Z2[r]["record"]["null"] for r in rp],
               yerr=[[Z2[r]["record"]["null"] - Z2[r]["record"]["lo"] for r in rp],
                     [Z2[r]["record"]["hi"] - Z2[r]["record"]["null"] for r in rp]],
               fmt="o-", capsize=4, color="tab:gray", label="record-shuffle")
ax[2].errorbar(rp, [Z2[r]["stratum"]["null"] for r in rp],
               yerr=[[Z2[r]["stratum"]["null"] - Z2[r]["stratum"]["lo"] for r in rp],
                     [Z2[r]["stratum"]["hi"] - Z2[r]["stratum"]["null"] for r in rp]],
               fmt="s-", capsize=4, color="tab:green", label="within-stratum")
ax[2].axhline(0.5, color="k", lw=1.0)
ax[2].axhline(REF["null_s2"], ls="--", color="tab:red", lw=1.0)
ax[2].annotate(f"Q7-S' null={REF['null_s2']}", (max(rp), REF["null_s2"]),
               fontsize=7, ha="right", va="bottom", color="tab:red")
ax[2].set_xscale("log", base=2); ax[2].set_xticks(rp)
ax[2].set_xticklabels([str(r) for r in rp])
ax[2].set_xlabel("permutations per record")
ax[2].set_ylabel("null matched-AUROC")
ax[2].set_title("Z2 : where did 0.5090 come from?", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q7z_direct_test", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⚠️")
for g in ("Z0", "Z1", "Z2", "Z3", "Z4"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if ok_("Z1"):
    run.log("  ★★★ **탈감쇠 모형이 검정을 통과했다 — λ 이득이 효용으로 전환된다.**")
    run.log(f"     최량 w = {BEST_S3:.0f}ms(관측) · 실측 오차 하 최량 w = {best_w:.0f}ms(Z3b)")
    run.log("     → **Q7-Y 진행.** 단 설계는 두 팔이다:")
    run.log("       팔 A 구간 최댓값 정규화 상호상관 · 팔 B **그 레코드 N 비트 PR 중앙에")
    run.log("       고정**한 정규화 상호상관(검출기·최댓값 **둘 다 불필요**)")
    run.log("       ★ 잡음 지배면 **최댓값은 잡음의 최댓값을 고른다**(양의 편의 · P 부재")
    run.log("         28.8% 비트에서 최악) — 좋은 일은 적분이 하고 max 는 위험만 얹는다")
    run.log("       ★ 팔 B ≈ 팔 A 면 **위치 정보 자체가 무용지물**이고, 분절기 사슬 전체가")
    run.log("         불필요했다는 뜻이다 — 큰 단순화다")
    run.log("       ★ **ST-T 구간 최댓값**을 선택 편의 바닥으로 같이 잰다")
elif no_("Z1"):
    run.log("  ⛔⛔ **탈감쇠 모형 기각 — λ 를 올려도 효용은 안 움직인다.**")
    run.log("     X6 의 상한 전부 무효 · 「31/63」 폐기 · **「자가 병목」 결론 재검토**")
    run.log("     → Q7-Y 를 짓지 않는다. 형태 갈래 종결을 검토한다")
else:
    run.log("  ⚠️ **λ 이득이 효용으로 전환되는지 이 표본에서 확인되지 않는다.**")
    run.log(f"     차 {DIFF['Z1']['diff']:+.4f} [{DIFF['Z1']['dlo']:+.4f}, "
            f"{DIFF['Z1']['dhi']:+.4f}] 가 0 을 덮는다 → X6 의 상한을 **예측이 아니라**")
    run.log("     **미검정**으로 표시하고,")
    run.log("     Q7-Y 는 **팔 B(고정 위치) 우선**으로 최소 설계만 짓는다")
run.log("")
run.log("  ▸ 필요표본은 **두 값 병기** — null 옛 " + str(REF["need"]["null_old"])
        + " · 새 " + str(REF["need"]["null_new"]) + f" (Z2 {VERD.get('Z2','?')})")
run.log("  ▸ 이 런은 **새 데이터를 쓰지 않았고 SVEB 질문에도 답하지 않는다** — 모형을 쟀다")
run.log("  ▸ 위 **결론 검산표**(Z5)가 각 문장의 미검정 가정을 나열한다 — 인용 전에 읽어라")

run.finish({
    "exp_id": "quest46_q7z_direct_test",
    "metric": "excess_diff_w40_minus_w0",
    "value": float(DIFF["Z1"]["diff"]),
    "passed": bool(ok_("Z0") and ok_("Z1")),
    "summary": ("탈감쇠 모형 직접 검정 — 적분 분자로 S3 를 다시 재서 λ 이득이 효용으로 "
                "전환되는지 본다. 덤으로 null 기전·128Hz 라운딩·λ 짝 맞추기·결론 검산표."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "Z0": CONFIG.get("Z0", {}), "Z1": CONFIG.get("Z1", {}), "Z2": CONFIG.get("Z2", {}),
    "Z3": CONFIG.get("Z3", {}), "Z3b": CONFIG.get("Z3b", {}), "Z4": CONFIG.get("Z4", {}),
    "Z5": CONFIG.get("Z5", []), "best_w": float(best_w), "lam_128": float(LAM_128),
    "but": CONFIG.get("but", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step deattenuation-test`")